# Part 8 — Machine Learning Forecasting

**PyTimeSeries user guide** · *Gradient boosting and global (cross-series) models.*

> Uses the `pytimeseries` package on the shared commodity-price dataset.

In [1]:
import warnings; warnings.simplefilter("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pytimeseries as pt
plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})

# Shared dataset: 356 months of global commodity prices (1992-2021)
raw = pd.read_csv("monthly_price.csv")
dates = pd.to_datetime(raw["YEAR"].astype(str) + "-" + raw["MONTH"], format="%Y-%b")
df = raw.drop(columns=["ID", "YEAR", "MONTH", "TIME"]); df.index = dates
df.index.name = "date"; df = df.asfreq("MS")
y = df["Food_Price_Index"]
print("pytimeseries", pt.__version__, "| series:", list(df.columns))

pytimeseries 0.5.0 | series: ['Urea_Price', 'MP_Price', 'TSP_Price', 'Maize_Price', 'Rice_Price', 'Wheat_Price', 'Food_Price_Index', 'Energy_Price_Index', 'VIX_Index']


In [2]:
cv = pt.ExpandingWindow(h=12, n_splits=5)
lgbm = pt.make_reduction(pt.ml.LightGBM(), strategy='recursive', lags=12)
pt.evaluate(lgbm, y, cv=cv, metrics=('MASE','RMSE'), sp=12).summary

metric,MASE,RMSE
model,,
Drift,0.7496,7.3852
Naive,0.7579,7.4806
SeasonalNaive(sp=12),0.9055,8.8899
RecursiveReductionForecaster,1.1215,11.2253


In [3]:
# ONE global model across all nine series
long = df.reset_index().melt(id_vars='date', var_name='unique_id', value_name='y')
long = long.rename(columns={'date': 'ds'})
gf = pt.GlobalForecaster(pt.ml.LightGBM(), lags=12).fit(long)
fc = gf.predict(6)
print('global fit ->', fc['unique_id'].nunique(), 'series,', len(fc), 'rows')
fc.head(3)

global fit -> 9 series, 54 rows


,unique_id,ds,forecast
0,Urea_Price,2021-09-01,444.322293
1,Urea_Price,2021-10-01,449.972991
2,Urea_Price,2021-11-01,447.269032


**Takeaway.** `pt.ml.LightGBM()` composes into `make_reduction` (local) or `GlobalForecaster` (one model, many series) — the ML-reduction payoff.